# Notebook 2: Circuit-Based Quantum Computing

This notebook is part of the hands-on materials for Session 2 of the FDP. We cover:
1. **Elementary Gates** (Single- and Two-qubit unitaries in Qiskit and PennyLane).
2. **Toy Example 1**: Bell State Preparation.
3. **Toy Example 2**: Superdense Coding.
4. **Toy Example 3**: The Deutsch Algorithm (1-bit phase kickback).

We will implement all concepts in both **Qiskit** and **PennyLane** for comparative learning.

## Section 1: Elementary Gates and Matrices

Let's review the matrix forms of the single-qubit gates ($X$, $Y$, $Z$, $H$, $S$, $T$) and construct circuits using them.

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

# 1. Define single-qubit gates in Qiskit and print their matrices
gates = ['x', 'y', 'z', 'h', 's', 't']
for g in gates:
    qc = QuantumCircuit(1)
    getattr(qc, g)(0)
    matrix = Operator(qc).data
    print(f"Gate: {g.upper()}")
    print(np.round(matrix, 4))
    print("-" * 30)

Gate: X
[[0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j]]
------------------------------
Gate: Y
[[0.+0.j 0.-1.j]
 [0.+1.j 0.+0.j]]
------------------------------
Gate: Z
[[ 1.+0.j  0.+0.j]
 [ 0.+0.j -1.+0.j]]
------------------------------
Gate: H
[[ 0.7071+0.j  0.7071+0.j]
 [ 0.7071+0.j -0.7071+0.j]]
------------------------------
Gate: S
[[1.+0.j 0.+0.j]
 [0.+0.j 0.+1.j]]
------------------------------
Gate: T
[[1.    +0.j     0.    +0.j    ]
 [0.    +0.j     0.7071+0.7071j]]
------------------------------


### Two-qubit Gates
We check CNOT and SWAP gate matrices in Qiskit.

In [2]:
# CNOT Gate (Control = 0, Target = 1)
qc_cnot = QuantumCircuit(2)
qc_cnot.cx(0, 1)
print("CNOT Matrix (Qiskit ordering, little-endian):")
print(Operator(qc_cnot).data)

print("\nSWAP Matrix:")
qc_swap = QuantumCircuit(2)
qc_swap.swap(0, 1)
print(Operator(qc_swap).data)

CNOT Matrix (Qiskit ordering, little-endian):
[[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]]

SWAP Matrix:
[[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]]


## Section 2: Bell State Preparation

We prepare the Bell state $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ in both PennyLane and Qiskit.

In [3]:
# --- Qiskit implementation ---
from qiskit_aer import AerSimulator

qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure_all()

print("Qiskit Bell Circuit:")
print(qc_bell.draw(output='text'))

sim = AerSimulator()
counts = sim.run(qc_bell, shots=1000).result().get_counts()
print("Qiskit Bell Counts:", counts)

Qiskit Bell Circuit:
        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 
Qiskit Bell Counts: {'11': 513, '00': 487}


In [4]:
# --- PennyLane implementation ---
import pennylane as qml

dev_bell = qml.device("default.qubit", wires=2, shots=1000)

@qml.qnode(dev_bell)
def bell_state_pl():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.counts()

print("PennyLane Bell Circuit:")
print(qml.draw(bell_state_pl)())

counts_pl = bell_state_pl()
print("\nPennyLane Bell Counts (Default binary mapping):")
print(counts_pl)

PennyLane Bell Circuit:
0: ──H─╭●─┤  Counts
1: ────╰X─┤  Counts

PennyLane Bell Counts (Default binary mapping):
{np.str_('00'): np.int64(510), np.str_('11'): np.int64(490)}


C:\Users\Shashank Gupta\SRM-AP_Sessions_July2026\.venv\Lib\site-packages\pennylane\devices\device_api.py:207: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


## Section 3: Superdense Coding

Superdense coding allows Alice to transmit two classical bits of information ($ab \in \{00, 01, 10, 11\}$) to Bob by sending only a single qubit, using a pre-shared entangled Bell pair.

In [5]:
def superdense_coding_qiskit(message):
    # Prepare shared Bell State on Q0 and Q1
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    
    qc.barrier() # Visual separation
    
    # Alice encodes her message on Q0 (her half of the Bell pair)
    # message is '00', '01', '10', or '11'
    if message[1] == '1':
        qc.x(0) # Apply X
    if message[0] == '1':
        qc.z(0) # Apply Z
        
    qc.barrier()
    
    # Alice sends Q0 to Bob. Bob measures both Q0 and Q1 in the Bell basis:
    qc.cx(0, 1)
    qc.h(0)
    qc.measure_all()
    
    return qc

# Run for all 4 possible messages
for msg in ['00', '01', '10', '11']:
    circuit = superdense_coding_qiskit(msg)
    res = sim.run(circuit, shots=1).result()
    received = list(res.get_counts().keys())[0]
    # Note: Qiskit registers are in reverse order (q1 q0), so we adjust representation
    print(f"Alice sent: {msg} | Bob measured (q1 q0): {received} | Success: {msg == received}")

Alice sent: 00 | Bob measured (q1 q0): 00 | Success: True
Alice sent: 01 | Bob measured (q1 q0): 10 | Success: False
Alice sent: 10 | Bob measured (q1 q0): 01 | Success: False
Alice sent: 11 | Bob measured (q1 q0): 11 | Success: True


### Superdense Coding in PennyLane

In [6]:
dev_sd = qml.device("default.qubit", wires=2)

@qml.qnode(dev_sd)
def superdense_coding_pl(message):
    # 1. Shared Bell Pair
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    
    # 2. Alice's Encoding
    if message[1] == '1':
        qml.PauliX(wires=0)
    if message[0] == '1':
        qml.PauliZ(wires=0)
        
    # 3. Bob's Decoding
    qml.CNOT(wires=[0, 1])
    qml.Hadamard(wires=0)
    
    return qml.probs(wires=[0, 1])

for msg in ['00', '01', '10', '11']:
    probs = superdense_coding_pl(msg)
    decoded_int = np.argmax(probs)
    decoded_bin = f"{decoded_int:02b}"
    print(f"Alice sent: {msg} | Bob decoded: {decoded_bin} | Probs: {np.round(probs, 4)}")

Alice sent: 00 | Bob decoded: 00 | Probs: [1. 0. 0. 0.]
Alice sent: 01 | Bob decoded: 01 | Probs: [0. 1. 0. 0.]
Alice sent: 10 | Bob decoded: 10 | Probs: [0. 0. 1. 0.]
Alice sent: 11 | Bob decoded: 11 | Probs: [0. 0. 0. 1.]


## Section 4: The Deutsch Algorithm (1-bit Phase Kickback)

The Deutsch algorithm determines whether a black-box 1-bit function $f(x)$ is constant ($f(0) = f(1)$) or balanced ($f(0) \neq f(1)$) in a single query. 
It uses **phase kickback** by putting the target helper qubit in the $|-\rangle$ state.

Let's implement the 4 possible oracles $U_f$ and run the algorithm.

In [7]:
# Define the 4 possible black-box oracles in Qiskit
def get_oracle_qiskit(oracle_type):
    qc = QuantumCircuit(2)
    if oracle_type == "constant_0":
        pass # f(x) = 0
    elif oracle_type == "constant_1":
        qc.x(1) # f(x) = 1
    elif oracle_type == "balanced_identity":
        qc.cx(0, 1) # f(x) = x
    elif oracle_type == "balanced_not":
        qc.x(1)
        qc.cx(0, 1) # f(x) = not x
    return qc

# Construct and run the Deutsch Algorithm in Qiskit
def run_deutsch_qiskit(oracle_type):
    qc = QuantumCircuit(2, 1) # 2 qubits, 1 classical bit for measuring Q0
    
    # Initialize target qubit (Q1) in |->
    qc.x(1)
    qc.h(1)
    
    # Initialize control qubit (Q0) in |+>
    qc.h(0)
    
    # Apply oracle
    oracle = get_oracle_qiskit(oracle_type)
    qc.compose(oracle, inplace=True)
    
    # Apply final Hadamard on control (Q0) to measure in X basis
    qc.h(0)
    
    # Measure Q0
    qc.measure(0, 0)
    return qc

print("Deutsch Algorithm Outputs:")
for o_type in ["constant_0", "constant_1", "balanced_identity", "balanced_not"]:
    qc_d = run_deutsch_qiskit(o_type)
    counts_d = sim.run(qc_d, shots=100).result().get_counts()
    outcome = list(counts_d.keys())[0]
    # 0 = Constant, 1 = Balanced
    classification = "Constant" if outcome == '0' else "Balanced"
    print(f"Oracle: {o_type:18} | Measured Q0: {outcome} ({classification})")

Deutsch Algorithm Outputs:
Oracle: constant_0         | Measured Q0: 0 (Constant)
Oracle: constant_1         | Measured Q0: 0 (Constant)
Oracle: balanced_identity  | Measured Q0: 1 (Balanced)
Oracle: balanced_not       | Measured Q0: 1 (Balanced)


### Deutsch Algorithm in PennyLane

In [8]:
dev_deutsch = qml.device("default.qubit", wires=2)

def apply_oracle_pl(oracle_type):
    if oracle_type == "constant_0":
        pass
    elif oracle_type == "constant_1":
        qml.PauliX(wires=1)
    elif oracle_type == "balanced_identity":
        qml.CNOT(wires=[0, 1])
    elif oracle_type == "balanced_not":
        qml.PauliX(wires=1)
        qml.CNOT(wires=[0, 1])

@qml.qnode(dev_deutsch)
def run_deutsch_pl(oracle_type):
    # Initialize Q1 in |->
    qml.PauliX(wires=1)
    qml.Hadamard(wires=1)
    
    # Initialize Q0 in |+>
    qml.Hadamard(wires=0)
    
    # Apply Oracle
    apply_oracle_pl(oracle_type)
    
    # Measure Q0 in X-basis
    qml.Hadamard(wires=0)
    return qml.probs(wires=0)

print("PennyLane Deutsch Algorithm Outputs:")
for o_type in ["constant_0", "constant_1", "balanced_identity", "balanced_not"]:
    probs = run_deutsch_pl(o_type)
    # probs[0] = P(0) -> Constant; probs[1] = P(1) -> Balanced
    classification = "Constant" if probs[0] > 0.9 else "Balanced"
    print(f"Oracle: {o_type:18} | P(0): {probs[0]:.2f} | P(1): {probs[1]:.2f} ({classification})")

PennyLane Deutsch Algorithm Outputs:
Oracle: constant_0         | P(0): 1.00 | P(1): 0.00 (Constant)
Oracle: constant_1         | P(0): 1.00 | P(1): 0.00 (Constant)
Oracle: balanced_identity  | P(0): 0.00 | P(1): 1.00 (Balanced)
Oracle: balanced_not       | P(0): 0.00 | P(1): 1.00 (Balanced)
